In [ ]:
## Defining packages
import os
import shutil
import pathlib
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path
import piexif
import re

## Defining Static Variables
datetime_original_tag = 36867
datetime_digitized_tag = 36868
datetime_tag = 306

In [ ]:
source_base = "D:/lara/Snapchat/snapchat_downloads_2022_2023_renamed/"
destination_base = "D:/lara/Snapchat with EXIF/"

In [ ]:
def filename_timestamp_splitter(filename):
    
    # extract only digits
    digits = re.sub(r"\D", "", filename)

    # must be exactly 14 digits
    if len(digits) < 14:
        return False

    digits = digits[:14]
    return digits

In [ ]:
def format_exif_timestamp(datetime_string):
    
    parsed = datetime.strptime(datetime_string, "%Y%m%d%H%M%S")
    exif_format = parsed.strftime("%Y:%m:%d %H:%M:%S")
    
    return exif_format

In [ ]:
def contains_real_datetime(filename):
    
    # extract only digits
    digits = re.sub(r"\D", "", filename)

    # must be exactly 14 digits
    if len(digits) < 14:
        return False

    digits = digits[:14]

    try:
        datetime.strptime(digits, "%Y%m%d%H%M%S")
        return True
    except ValueError:
        return False

In [ ]:
def has_exif_datetime(image_path):
    
    img = Image.open(image_path)
    exif_data = img._getexif()
    
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    img_datetime = exif_data.get(datetime_tag)

    true_datetime = original_datetime or digitized_datetime or img_datetime
    
    if true_datetime:
        return True
    else:
        return False

# Steps

1. check if filename has valid datetime name (if not = ignore, no way to set exif data)
2. check if there is exif data for main datetime original (if there is = ignore, no reason to add datetime data)
   - but check first that it matches the filename, flag it if it doesnt
3. check if there is exif data for other datetime values (if there is, must do some validation)
   - if there is no exif datetime data at all -> easy = use filename datetime
   - if there is a datetime value -> check that it matches the filename
      - if it does match the filename -> good to set all datetimes to that value
      - if it does not match the filename -> ignore and flag the file as contradictory
4. copy image to destination
6. add all datetime values with the one found in filename (if previous checks have passed)
7. do some validation that the datetime value has changed and maybe some validation that the rest of the exif hasn't been altered? if not overkill?
8. once the file has been moved (check it exists) and there's a datetime original value, delete the original image from the source path

Note: This code only works reliably for JPG/JPEG files.

In [ ]:
contradictary_files = {}

folder = Path(source_base)
for file in folder.rglob("*"):
    
    suffix = Path(str(file)).suffix
    if suffix.upper() not in ['.JPG', '.JPEG']:
        continue
    
    # Defining file-specific variables
    file_source_path = str(file)
    filename = Path(file_source_path).name
    img = Image.open(file_source_path)
    exif_data = img._getexif()
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    plain_datetime = exif_data.get(datetime_tag)
    filename_datetime = filename_timestamp_splitter(filename[:17])
    
    print("Reading: ", file_source_path)
    
    # Step 1: Ignore if there is no datetime found in the filename
    if not contains_real_datetime(filename[:17]):
        print("   Step 1: Failed (filename does not contain datetime data)")
        continue

    print("   Step 1: Passed")
    filename_datetime_formatted = format_exif_timestamp(filename_datetime)
    
    
    # Step 2: Ignore if there is already a datetime found in the EXIF data (but with consolidation checks)
    if has_exif_datetime(file_source_path):
        
        print("   Step 2: Failed (EXIF data already contains datetime data)")
        
        # Step 2.5: Flag the file if the filename datetime is different from the EXIF Datetime original
        if original_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = original_datetime
            print("   Step 2.5: Failed")

        continue
    
    print("   Step 2: Passed")
    
    # Step 3: Check that other exif datetime values don't differ from the filename datetime
    if digitized_datetime or plain_datetime:
        other_exif_datetime = digitized_datetime or plain_datetime
        if other_exif_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = other_exif_datetime
            print("   Step 3: Failed (Contradictory dates found)")
            continue
    
    print("   Step 3: Passed")
            
    # Happy that the filename will be the datetime value for the exif data
    final_exif_datetime = filename_datetime_formatted
    print("   Finalised datetime string = " + final_exif_datetime)
    
    # Step 4: Copy image to the destination
    file_dest_path = destination_base + filename
    try:
        shutil.copyfile(file_source_path, file_dest_path)
    except:
        print("Error copying file: " + file_path)
        print("   Step 4: Failed (Error copying file: " + file_path + ")")
        continue
    
    print("   Step 4: Passed")

    # Step 5: Add datetime values to exif data if they don't already exist
    with Image.open(file_dest_path) as new_img:
        new_exif_data = new_img._getexif()
        new_exif_dict = piexif.load(new_img.info.get("exif", b""))

        new_exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal]= final_exif_datetime

        try:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized]
        except:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized] = final_exif_datetime

        try:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime]
        except:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime] = final_exif_datetime

        exif_bytes = piexif.dump(new_exif_dict)

        new_img.save(file_dest_path, exif=exif_bytes)
        img.close()
        new_img.close()
    
    print("   Step 5: Passed")
    
    # Step 6: Validation
    if Path(file_dest_path).exists():
        
        with Image.open(file_dest_path) as verify_img:
            new_exif_data = verify_img._getexif()

        if new_exif_data.get(datetime_original_tag):
            os.remove(file_source_path)
        else:
            print("Datetime data not found in EXIF:", file_dest_path)
            continue
    else:
        print("Copy failed, not deleting:", source)
        continue
    


In [ ]:
contradictary_files